In [ ]:
pip install -r requirements.txt

In [ ]:
import Prompts as prompts


In [ ]:
import importlib
importlib.reload(prompts)

In [2]:
import numpy as np
import pandas as pd
from langchain_ollama import OllamaLLM
from langchain_core.prompts import PromptTemplate
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

In [3]:
from Prompts.prompts_hotels import Config

In [4]:
# Load your CSV
df_new = pd.read_csv("/home/sohy47ma/ReviewProject_new/fake_reviews_prediction/Datasets/Hotel_Human_VS_HumanFake_relabelled.csv")
reviews = df_new["text"].tolist()

In [5]:
prompt = PromptTemplate(
    input_variables=["text"], # the placeholder to be replaced based on prompt template
    template=Config.zero_shot_prompt_template
)

# Load small model
llm = OllamaLLM(model=Config.model)

# Create chain using pipe operator (modern LangChain syntax)
chain = prompt | llm

In [6]:
print(Config.zero_shot_prompt_template)


    You are an expert at detecting fake and real reviews.

    Task: Classify the following review of a hotel that is listed on a website as either "fake" or "real". Analyze the language and structure of the review,
	It should not be overly generic or templeted language,should not have sales-like wording.

    Review: "{text}"

    Classify this review as either "fake" or "real" (respond with only one word):


In [ ]:
apptainer --version

In [ ]:
import os
os.environ["OLLAMA_HOST"] = "http://localhost:11400"

In [ ]:
# Classify each review
results = []
for r in reviews:
    result = chain.invoke({"text": r})
    # Clean the output - extract only "truthful" or "deceptive"
    label = result.strip().lower()
    # Extract first word if model adds extra text
    if " " in label:
        label = label.split()[0]
    # Remove any punctuation
    label = label.strip('.,!?;:')
    print(f"Raw output: {result} -> Cleaned: {label}")
    results.append(label)

In [ ]:
# Classify each review
results = []

for r in reviews:
    result = chain.invoke({"text": r})
    
    label = result.strip().lower()
    
    # Keep only valid labels
    if "fake" in label:
        cleaned_label = "fake"
    elif "real" in label:
        cleaned_label = "real"
    else:
        cleaned_label = "fake"
    cleaned_label = cleaned_label.strip('.,!?;:')
    print(f"Raw output: {result} -> Cleaned: {cleaned_label}")
    results.append(cleaned_label)


In [9]:
df_new.head(10)


,Binary_label,Category,source,text,is_synthetic
0,fake,Home_and_Kitchen,amazon,Air pressure was utter crap. I kept the lid o...,1
1,fake,Home_and_Kitchen,amazon,"Not what I expected, smaller than expected, bu...",1
2,fake,Home_and_Kitchen,amazon,We like a toaster that doesn't have the long h...,1
3,fake,Home_and_Kitchen,amazon,Love these pans they work great. The only prob...,1
4,fake,Home_and_Kitchen,amazon,I love the print on the back and the finish. I...,1
5,fake,Home_and_Kitchen,amazon,Bought this on a whim and it was the best purc...,1
6,fake,Home_and_Kitchen,amazon,"I admit I was leary of the suction, but I did ...",1
7,fake,Home_and_Kitchen,amazon,Purchased one as a gift and it was a very nice...,1
8,fake,Home_and_Kitchen,amazon,I wanted something to hold a bunch of pieces o...,1
9,fake,Home_and_Kitchen,amazon,I'm so happy I found this set and am very plea...,1


In [8]:
# def clean_label(text):
#     return text.strip().lower().split()[0].strip('.,!?;:')
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

def evaluate_model(df, results):
    # Get true labels from the dataframe
    #y_true = df["deceptive"].tolist()
    y_true = df["Binary_label"].tolist()
    y_pred = results

    # Calculate metrics
    accuracy = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='weighted', pos_label=None)
    conf_matrix = confusion_matrix(y_true, y_pred)

    return accuracy, f1, conf_matrix

In [ ]:
y_true = df_new["Binary_label"].tolist()

In [ ]:
for label in results:
    print(label)

In [9]:
accuracy, f1, conf_matrix = evaluate_model(df_new, results)
print(Config.model)
print("=" * 50)
print("ZERO-SHOT PROMPTING RESULTS HOTELS - HUMAN VS HUMAN FAKE : LLAMA 3.2 3B")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

llama3.2:3b
ZERO-SHOT PROMPTING RESULTS HOTELS - HUMAN VS HUMAN FAKE : LLAMA 3.2 3B
Accuracy: 0.5062814070351759

F1 Score: 0.35855416943480717

Confusion Matrix:
[[ 21 775]
 [ 11 785]]


In [9]:
# Results with LLAMA 3.1 8B
print(Config.model)
accuracy, f1, conf_matrix = evaluate_model(df_new, results)

print("=" * 50)
print("ZERO-SHOT PROMPTING RESULTS HOTELS - HUMAN VS HUMAN FAKE : LLAMA 3.1 8B")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

llama3.1:8b
ZERO-SHOT PROMPTING RESULTS HOTELS - HUMAN VS HUMAN FAKE : LLAMA 3.1 8B
Accuracy: 0.5408291457286433

F1 Score: 0.43348091606769495

Confusion Matrix:
[[ 84 712]
 [ 19 777]]


## One-Shot Prompting
Using one example to guide the model's classification

In [5]:
llm = OllamaLLM(model=Config.model)

In [6]:
one_shot_prompt = PromptTemplate(
    input_variables=["text"],
    template = Config.one_shot_prompt_template
)
# Create one-shot chain
one_shot_chain = one_shot_prompt | llm

In [7]:
print(Config.one_shot_prompt_template)


     You are an expert at detecting fake and real reviews.

    Task: Classify the following review of a hotel that is listed on a website as either "real" or "fake". Analyze the language and structure of the review,
	It should not be overly generic or templeted language,should not have sales-like wording and lacks details or specificity.

    Example:
    Review: "As a former Chicagoan, I'm appalled at the Amalfi Hotel Chicago. First of all, I was expecting luxury and hospitality, neither of which I received. There's an Experience Designer who is supposed to be like a 'personal concierge,' but my experience with my ED was terrible. I felt like he was trying to pressure me into staying more days than I wanted to. Not only that, but I couldn't understand what he was saying most of the time because he was talking so fast. When I finally got to my room, I was disappointed with the quality of the furniture and the room's cleanliness. I had to ask for a maid to come and give me clean towel

In [ ]:
# Classify each review
one_shot_results = []

for r in reviews:
    result = one_shot_chain.invoke({"text": r})
    
    label = result.strip().lower()
    
    # Keep only valid labels
    if "fake" in label:
        cleaned_label = "fake"
    elif "real" in label:
        cleaned_label = "real"
    else:
        cleaned_label = "fake"
    cleaned_label = cleaned_label.strip('.,!?;:')
    print(f"Raw output: {result} -> Cleaned: {cleaned_label}")
    one_shot_results.append(cleaned_label)


In [ ]:
df_new.head(10)

In [10]:
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

def evaluate_model(df, results):
    # Get true labels from the dataframe
    #y_true = df["deceptive"].tolist()
    y_true = df["Binary_label"].tolist()
    y_pred = results

    # Calculate metrics
    accuracy = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='weighted', pos_label=None)
    conf_matrix = confusion_matrix(y_true, y_pred)

    return accuracy, f1, conf_matrix

In [11]:
#One shot results HOTEL HUMAN VS HUMAN FAKE with LLAMA 3.2 3B
accuracy_one_shot, f1_one_shot, conf_matrix_one_shot = evaluate_model(df_new, one_shot_results)

print("=" * 50)
print("ONE-SHOT PROMPTING RESULTS HOTEL HUMAN VS HUMAN FAKE WITH LLAMA 3.2 3B")
print("=" * 50)
print("Accuracy:", accuracy_one_shot)
print("\nF1 Score:", f1_one_shot)
print("\nConfusion Matrix:")
print(conf_matrix_one_shot)
print("=" * 50)

ONE-SHOT PROMPTING RESULTS HOTEL HUMAN VS HUMAN FAKE WITH LLAMA 3.2 3B
Accuracy: 0.5559045226130653

F1 Score: 0.5494188711452399

Confusion Matrix:
[[347 449]
 [258 538]]


In [15]:
#One shot results HOTEL HUMAN VS HUMAN FAKE with LLAMA 3.1 8B
accuracy_one_shot, f1_one_shot, conf_matrix_one_shot = evaluate_model(df_new, one_shot_results)
print(Config.model)
print("=" * 50)
print("ONE-SHOT PROMPTING RESULTS HOTEL HUMAN VS HUMAN FAKE WITH LLAMA 3.1 8B")
print("=" * 50)
print("Accuracy:", accuracy_one_shot)
print("\nF1 Score:", f1_one_shot)
print("\nConfusion Matrix:")
print(conf_matrix_one_shot)
print("=" * 50)

llama3.1:8b
ONE-SHOT PROMPTING RESULTS HOTEL HUMAN VS HUMAN FAKE WITH LLAMA 3.1 8B
Accuracy: 0.6224874371859297

F1 Score: 0.6208380705854194

Confusion Matrix:
[[443 353]
 [248 548]]


## Few-Shot Prompting
Using multiple examples to guide the model's classification

In [12]:
llm = OllamaLLM(model=Config.model)

In [13]:
few_shot_prompt = PromptTemplate(
    input_variables=["text"],
    template=Config.few_shot_prompt_template
)

# Create few-shot chain
few_shot_chain = few_shot_prompt | llm

In [14]:
print(Config.few_shot_prompt_template)


    You are an expert at detecting real and fake reviews.


    Task: Classify the following review of a hotel that is listed on a website as either "real" or "fake". Analyze the language and structure of the review,
	It should not be overly generic or templeted language,should not have sales-like wording and lacks details or specificity.

    Here are some examples:

    Examples:
    Example 1:
    Review: "As a former Chicagoan, I'm appalled at the Amalfi Hotel Chicago. First of all, I was expecting luxury and hospitality, neither of which I received. There's an Experience Designer who is supposed to be like a 'personal concierge,' but my experience with my ED was terrible. I felt like he was trying to pressure me into staying more days than I wanted. Not only that, but I couldn't understand what he was saying most of the time because he was talking so fast. When I finally got to my room, I was disappointed with the quality of the furniture and the room's cleanliness. I had to ask 

In [ ]:
# Classify each review
few_shot_results = []

for r in reviews:
    result = few_shot_chain.invoke({"text": r})
    
    label = result.strip().lower()
    
    # Keep only valid labels
    if "fake" in label:
        cleaned_label = "fake"
    elif "real" in label:
        cleaned_label = "real"
    else:
        cleaned_label = "fake"
    cleaned_label = cleaned_label.strip('.,!?;:')
    print(f"Raw output: {result} -> Cleaned: {cleaned_label}")
    few_shot_results.append(cleaned_label)


In [16]:
accuracy_few_shot, f1_few_shot, conf_matrix_few_shot = evaluate_model(df_new, few_shot_results)

print("=" * 50)
print("FEW-SHOT PROMPTING RESULTS HUMAN VS HUMAN FAKE FOR LLAMA 3.2:3B")
print("=" * 50)
print("Accuracy:", accuracy_few_shot)
print("\nF1 Score:", f1_few_shot)
print("\nConfusion Matrix:")
print(conf_matrix_few_shot)
print("=" * 50)

FEW-SHOT PROMPTING RESULTS HUMAN VS HUMAN FAKE FOR LLAMA 3.2:3B
Accuracy: 0.5672110552763819

F1 Score: 0.5653918311471935

Confusion Matrix:
[[503 293]
 [396 400]]


In [20]:
# Results for few shot HUMAN VS HUMAN FAKE - llama 3.1 8b
accuracy_few_shot, f1_few_shot, conf_matrix_few_shot = evaluate_model(df_new, few_shot_results)
print(Config.model)
print("=" * 50)
print("FEW-SHOT PROMPTING RESULTS HUMAN VS HUMAN FAKE FOR LLAMA 3.1:8B")
print("=" * 50)
print("Accuracy:", accuracy_few_shot)
print("\nF1 Score:", f1_few_shot)
print("\nConfusion Matrix:")
print(conf_matrix_few_shot)
print("=" * 50)

llama3.1:8b
FEW-SHOT PROMPTING RESULTS HUMAN VS HUMAN FAKE FOR LLAMA 3.1:8B
Accuracy: 0.621859296482412

F1 Score: 0.6205922406967538

Confusion Matrix:
[[541 255]
 [347 449]]


## Compare All Approaches
Compare zero-shot, one-shot, and few-shot results

In [ ]:
# Compare all approaches
comparison_df = pd.DataFrame({
    'Approach': ['Zero-Shot', 'One-Shot', 'Few-Shot'],
    'Accuracy': [accuracy, accuracy_one_shot, accuracy_few_shot],
    'F1 Score': [f1, f1_one_shot, f1_few_shot]
})

print("=" * 60)
print("COMPARISON OF ALL PROMPTING APPROACHES")
print("=" * 60)
print(comparison_df.to_string(index=False))
print("=" * 60)

# Find best approach
best_idx = comparison_df['Accuracy'].idxmax()
best_approach = comparison_df.loc[best_idx, 'Approach']
best_accuracy = comparison_df.loc[best_idx, 'Accuracy']
print(f"\nBest Approach: {best_approach} (Accuracy: {best_accuracy:.4f})")